In [1]:
import pandas as pd
from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import LabelEncoder

# 1. Load Data
df = pd.read_excel('Individual Assignment Data File Electronic_sales.xlsx')
df['Is_Cancelled'] = (df['Order Status'] == 'Cancelled').astype(int)

# 2. Select Features
# We drop the Target variable (Order Status) and IDs. We only want 
# the algorithm to look at customer attributes and order details.
features = ['Age', 'Total Price', 'Unit Price', 'Quantity', 'Rating', 
            'Gender', 'Loyalty Member', 'Product Type', 'Payment Method', 'Shipping Type']

X = df[features].copy()

# 3. Preprocessing: Isolation Forest requires numerical inputs
# We use LabelEncoder to quickly convert text categories (like "Male"/"Female") to numbers (0/1).
for col in X.select_dtypes(include=['object']).columns:
    X[col] = LabelEncoder().fit_transform(X[col].astype(str))

X = X.fillna(0) # Safety check for missing data

# 4. Initialize and Train the Isolation Forest
# n_estimators: Number of trees to build.
# contamination: The percentage of data you suspect are outliers. 
#                Setting it to 0.05 means we want it to find the top 5% "weirdest" orders.
iso_forest = IsolationForest(n_estimators=100, contamination=0.05, random_state=42)

# fit_predict() outputs '1' for Normal data, and '-1' for Anomalies/Outliers.
df['Anomaly_Raw'] = iso_forest.fit_predict(X)

# 5. Clean up the output for easier reading
# Let's map '-1' (Anomaly) to '1', and '1' (Normal) to '0'
df['Is_Anomaly'] = df['Anomaly_Raw'].map({1: 0, -1: 1})

# 6. Evaluate: Do Anomalies correlate with Cancellations?
print("Total Anomalies Detected:\n", df['Is_Anomaly'].value_counts())

normal_cancel_rate = df[df['Is_Anomaly'] == 0]['Is_Cancelled'].mean()
anomaly_cancel_rate = df[df['Is_Anomaly'] == 1]['Is_Cancelled'].mean()

print(f"\nCancellation Rate for Normal Orders: {normal_cancel_rate:.2%}")
print(f"Cancellation Rate for Anomalous Orders: {anomaly_cancel_rate:.2%}")

Total Anomalies Detected:
 Is_Anomaly
0    19000
1     1000
Name: count, dtype: int64

Cancellation Rate for Normal Orders: 32.87%
Cancellation Rate for Anomalous Orders: 32.20%
